# Part 3.3 Online evaluators

After this notebook you can score live traffic automatically, with LLM-judge templates and with
a plain Python rule.

In [ ]:
from kea.config import setup

setup("kea-rebooking-local")

## See it

Offline evals (Part 2) ask "did my change break anything I already know about?". Online
evaluators ask "is it working right now?". A **rule** on the deployment's tracing project scores
each new trace as it arrives, so you find the cases you did not think to test.

We add three:

| Evaluator | Scores | Reads | Cost | `true` means |
|---|---|---|---|---|
| `pii_leakage` | the passenger's message and the agent's reply | each run | LLM call | a problem |
| `tool_selection` | the whole conversation's tool choices | each thread, once idle | LLM call | good tool use |
| `credit_without_approval` | `issue_credit` ran with no human approval | each run | free | a problem |

<img src="../../assets/screenshots/p3-03-online-evaluators.png" alt="the project's Evaluators tab listing the three rules, each with sampling 1.0." width="900">

<img src="../../assets/screenshots/p3-03-pii-score.png" alt="a trace where piileakage is true, with the judge's reasoning in the feedback comment." width="900">

To reproduce in the UI: project → **Evaluators** → **+ Evaluator** → **Create from template**.
Pick **PII Leakage** or **Tool Selection**, set the filter and sampling, and save.

## Build it

The two LLM templates are prompts from the open-source `openevals` package. Read one.

In [ ]:
from openevals.prompts import PII_LEAKAGE_PROMPT

print(PII_LEAKAGE_PROMPT[:700])

The third rule needs no model. `credit_without_approval` is a Python function that flags a run
where `issue_credit` executed without a resume payload in its inputs. LangSmith uploads this exact
source.

In [ ]:
import inspect

from kea.evals.online import perform_eval

print(inspect.getsource(perform_eval))

Test it on two hand-made runs before LangSmith runs it. The first has a credit and no approval.
The second resumed from an approval.

In [ ]:
credit = {"type": "tool", "name": "issue_credit", "status": "success"}
person = {"type": "human", "content": "Give me a $500 credit."}
messages = {"messages": [person, credit]}

print(perform_eval({"inputs": {"messages": [person]}, "outputs": messages}))
approved = {"resume": {"decisions": [{"type": "approve"}]}}
print(perform_eval({"inputs": approved, "outputs": messages}))

`kea.evals.online` turns the three evaluators into rules. Here is what it will send, built without
calling LangSmith.

In [ ]:
from kea.evals.online import build_rules

for rule in build_rules("<project-id>", {"id": "", "settings": {}}):
    kind = "code" if "code_evaluators" in rule.body else "LLM judge"
    scope = rule.body.get("filter") or f"group_by {rule.body['group_by']}"
    print(f"{rule.name:24} {kind:10} {scope:20} sampling {rule.body['sampling_rate']}")

Create them on the `kea-rebooking` project. This is `mise run eval:online`. It changes the
workspace, so it only runs when you set `CREATE_RULES = True`. The script is idempotent: run it
again to update the rules.

In [ ]:
# needs: deployment
import subprocess
import sys

from kea.config import REPO_ROOT

CREATE_RULES = False

if CREATE_RULES:
    subprocess.run(
        [sys.executable, "-m", "kea.evals.online", "--deployment", "kea-rebooking"],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("CREATE_RULES is False: no rules created. Set it to True once the deployment is Ready.")

Rules score **new** traces only, so send traffic after the rules exist. `scripts/seed_traffic.py`
runs a dozen passenger conversations: normal rebookings, a passenger who pastes a card number, and
$500 credit demands that it approves or rejects through the interrupt. The cell is commented out
so it is not re-run by accident.

In [ ]:
# !cd ../.. && mise run seed -- --url "$KEA_DEPLOYMENT_URL"

Open the project's Traces tab. `pii_leakage` and `credit_without_approval` fill in within a minute
or two. `tool_selection` takes longer, because a thread is only scored after it has been idle for
two minutes.

## Under the hood

<img src="../../assets/slides/adlc-6-traces.jpg" width="720">

- **A rule is a filter, a sampling rate, and an evaluator.** These rules score root runs at
  sampling 1.0. Keep that for a workshop. For real traffic, sample (0.05 to 0.2) or filter to the
  flows you care about, because an LLM judge costs a model call per trace.
- **No reference answer online.** Live traffic has no expected output, so online evaluators check
  properties: did it leak PII, did it pick sensible tools, did it skip the approval step.
- **`true` means a problem for safety templates.** `pii_leakage` is `true` when PII is found.
  `tool_selection` is the other way round: `true` means the tool choices were good. The
  [template table](https://docs.langchain.com/langsmith/evaluators#prebuilt-evaluator-templates)
  states what `true` means for each one.
- **Thread evaluators wait for idle.** `tool_selection` reads a whole conversation, so LangSmith
  waits until no new trace arrives for the project's idle time. The minimum is two minutes and the
  script sets it, so scores show up a few minutes after the last message. The default is ten.
- **The API cannot instantiate a template by id.** The UI copies the `openevals` prompt into a
  rule, and `kea.evals.online` does the same. Our PII rule reads the final reply instead of the
  whole output, because tool results carry booking data by design. Do not widen it to the full
  output.
- **Code evaluators run in a sandbox.** LangSmith calls `perform_eval(run)` with the run as a
  dict. Standard library only, no network. Online, a resumed run's `inputs` hold the `resume`
  payload, which is how `credit_without_approval` tells an approved credit from an unapproved one.
- **Remove the rules** with `python -m kea.evals.online --deployment kea-rebooking --delete`.

Docs: [Online evaluators](https://docs.langchain.com/langsmith/evaluators).

## Recap

- A rule is a filter, a sampling rate, and an evaluator on the tracing project.
- Use a template when an LLM judge fits. Use code when the rule is cheap to state.
- Read `true` carefully, and expect thread scores a few minutes late.

**Next:** [3.4 Close the loop](04_close_the_loop.ipynb)